In [1]:
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.datasets import RetrievalDataset, MassSpecDataset
from chemembed_transforms import ChemEmbedSpecTransform, ChemEmbedMolTransform, ChemBERTaMolTransform, MoLFormerMolTransform
from pathlib import Path
from ann_model_massspecgym import AnnRetrieval
from pytorch_lightning import Trainer
from massspecgym.models.base import Stage
import pandas as pd

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


In [ ]:
######### Embedding selection ##########

# Mol2Vec (300 dims)
# selected_molecular_embedding = ChemEmbedMolTransform(model_path='Mol2Vec_model/model_300dim.pkl')
# embedding_name = "mol2vec"
# mol_embedding_dim = 300

# ChemBERTa (768 dims)
selected_molecular_embedding = ChemBERTaMolTransform()
embedding_name = "chemberta"
mol_embedding_dim = 768

# # MoLFormer (768 dims)
# selected_molecular_embedding = MoLFormerMolTransform()
# embedding_name = "molformer"
# mol_embedding_dim = 768

####################

# Train dataset, without reference candidates
train_dataset = MassSpecDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)
data_module_train = MassSpecDataModule(dataset=train_dataset, batch_size=32)

data_module_train.prepare_data()
data_module_train.setup()

# Init model
model = AnnRetrieval(
    mol_embedding_dim=mol_embedding_dim,
    log_only_loss_at_stages=[Stage.TRAIN, Stage.VAL]
)

# Init trainer
trainer = Trainer(
    accelerator="auto",
    max_epochs=15,
    log_every_n_steps=1
)

# Validation before training
trainer.validate(model, datamodule=data_module_train)

# Train
trainer.fit(model, datamodule=data_module_train)

# Save model checkpoint
trainer.save_checkpoint(f"ANN_trained_{embedding_name}.ckpt")

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aa\.cache\huggingface\hub\models--ibm--MoLFormer-XL-both-10pct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pytho

KeyboardInterrupt: 

In [ ]:
# Standard test 
test_dataset = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)
data_module_test = MassSpecDataModule(dataset=test_dataset, batch_size=32)
data_module_test.setup(stage="test")

test_results = trainer.test(model, datamodule=data_module_test)
results_df = pd.DataFrame(test_results)
results_df.to_csv(f"masspecgym_ann_results_{embedding_name}.csv", index=False)

In [ ]:
# BONUS test with candidates (same molecular formula)
test_dataset_bonus = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
    candidates_pth='bonus'
)
data_module_test_bonus = MassSpecDataModule(dataset=test_dataset_bonus, batch_size=32)
data_module_test_bonus.setup(stage="test")

test_results_bonus = trainer.test(model, datamodule=data_module_test_bonus)
results_df_bonus = pd.DataFrame(test_results_bonus)
results_df_bonus.to_csv(f"masspecgym_ann_results_bonus_{embedding_name}.csv", index=False)